In [1]:
from qiskit.circuit import Parameter, QuantumCircuit, QuantumRegister, ClassicalRegister

from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector, Operator
from scipy.optimize import minimize 
from qiskit.circuit.library import QFT
from qiskit import transpile
from qiskit.circuit.library import UnitaryGate

import random
import matplotlib.pyplot as plt
import scipy.linalg as scl
import numpy as np
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator()
# backend=FakeKyiv()
# sampler = Sampler(backend = backend)
pm = generate_preset_pass_manager(backend=backend,optimization_level=2)

In [2]:


nb_qubits = 4

N = 2**nb_qubits
m = np.zeros((N,N))
for j in range(N):
    if j == N-1:
        break
    else:
       m[j,j+1] = -1 

for j in range(N):
    if j == N-1:
        break
    else:
       m[j+1,j] = -1 
for j in range(N):
   m[j,j] = 2 
m[0] = np.array([1]+ [0]*(N-1))
m[1,0] = 0

b = 0.25*np.array([0,1,1,1,1,1,1,1,1,1,1,1,1,2,0,0])
nb_qubits = 4

In [3]:
d,u = np.linalg.eig(m)

In [4]:
max(d)/min(d)

103.08686891981826

In [5]:
def U_b(nb_qubits):
    circ = QuantumCircuit(nb_qubits)
    circ.prepare_state(b)
    return circ
U = U_b(nb_qubits)
b = np.array(Statevector(U_b(nb_qubits)))
b

array([1.52655666e-16+1.80411242e-16j, 2.50000000e-01+2.11636264e-15j,
       2.50000000e-01+1.85962357e-15j, 2.50000000e-01+1.74860126e-15j,
       2.50000000e-01-1.05471187e-15j, 2.50000000e-01-1.30451205e-15j,
       2.50000000e-01-1.49880108e-15j, 2.50000000e-01-1.74860126e-15j,
       2.50000000e-01-2.19269047e-15j, 2.50000000e-01-1.87350135e-15j,
       2.50000000e-01-1.66533454e-15j, 2.50000000e-01-1.41553436e-15j,
       2.50000000e-01+1.41553436e-15j, 5.00000000e-01+2.92821323e-15j,
       0.00000000e+00+8.32667268e-17j, 0.00000000e+00+1.84889275e-32j])

In [6]:
def Hamiltonian(m):
    Ub = np.array(Operator(U_b(nb_qubits)))
    z = np.array([[1,0],
                 [0,-1]])
    I = np.array([[1,0],
                 [0,1]])
    def tensor(i,j,k,l):
        return np.kron(i,np.kron(j,np.kron(k,l)))
    M1 = (np.dot(np.dot(Ub,tensor(z,I,I,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,z,I,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,I,z,I)),np.conj(Ub.T))
          + np.dot(np.dot(Ub,tensor(I,I,I,z)),np.conj(Ub.T)))
    M = 0.5*np.dot(np.dot(np.conj(m.T),(tensor(I,I,I,I) - M1/nb_qubits)),m)

    return M
A = Hamiltonian(m)

In [7]:

U1=scl.expm(2**0*2*np.pi*1j*A) 
U2=scl.expm(2**1*2*np.pi*1j*A) 
U3=scl.expm(2**2*2*np.pi*1j*A) 
U4=scl.expm(2**3*2*np.pi*1j*A) 
U5=scl.expm(2**4*2*np.pi*1j*A) 
U6=scl.expm(2**5*2*np.pi*1j*A) 
U7=scl.expm(2**6*2*np.pi*1j*A) 
U8=scl.expm(2**7*2*np.pi*1j*A)
 
u1gate = UnitaryGate(U1)
u2gate = UnitaryGate(U2)
u3gate=UnitaryGate(U3)
u4gate=UnitaryGate(U4)
u5gate=UnitaryGate(U5)
u6gate=UnitaryGate(U6)
u7gate=UnitaryGate(U7)
u8gate=UnitaryGate(U8)

C_u1gate=u1gate.control()
C_u2gate=u2gate.control()
C_u3gate=u3gate.control()
C_u4gate=u4gate.control()
C_u5gate=u5gate.control()
C_u6gate=u6gate.control()
C_u7gate=u7gate.control()
C_u8gate=u8gate.control()
    

In [8]:
nb_qubits = 4
depth = 2
qubits = list(range(nb_qubits))
N = len(qubits)
nb_params = int(9*N*depth)
x_exact = np.linalg.solve(m,b)
x_exact = x_exact/np.linalg.norm(x_exact)
Parameters = np.array([random.random() for _ in range(0, nb_params)])
Parameters

array([0.67714423, 0.67772606, 0.69478165, 0.55647465, 0.43372933,
       0.53140017, 0.95426003, 0.2406321 , 0.87391001, 0.16124733,
       0.80867314, 0.02053739, 0.44051374, 0.55352571, 0.41202577,
       0.55372751, 0.22513093, 0.84591393, 0.68334541, 0.09165595,
       0.81821272, 0.95162033, 0.96876967, 0.22048062, 0.33422451,
       0.12304783, 0.58588176, 0.8858374 , 0.48802809, 0.5639669 ,
       0.36103037, 0.80103042, 0.44076896, 0.08439957, 0.13639747,
       0.37402348, 0.61862341, 0.56398701, 0.82877351, 0.65441412,
       0.91048932, 0.66216251, 0.91335801, 0.88933291, 0.6605191 ,
       0.18580787, 0.88938255, 0.52612077, 0.86152235, 0.78873492,
       0.09798084, 0.73049727, 0.92792005, 0.72704253, 0.07278459,
       0.6436027 , 0.4043491 , 0.86273309, 0.33801589, 0.54448908,
       0.79975683, 0.40208635, 0.95598093, 0.32243222, 0.80663165,
       0.32706378, 0.68432727, 0.07492522, 0.6198662 , 0.76271446,
       0.51945928, 0.67889528])

In [9]:
RMSE = []
Shots = [100, 1000, 10000, 100000]
for shots in Shots:
    Parameters = np.array([0.78640187, 0.62057986, 0.07464034, 0.91162889, 0.28657081,
       0.53032524, 0.78251657, 0.86584458, 0.34769372, 0.01056126,
       0.60912631, 0.72095658, 0.43075189, 0.48055686, 0.10813979,
       0.41466688, 0.41223045, 0.29993745, 0.06643231, 0.01785771,
       0.49337166, 0.50831322, 0.60358623, 0.91769195, 0.9760866 ,
       0.42879226, 0.8640231 , 0.65690642, 0.8285242 , 0.08401378,
       0.43854101, 0.68764187, 0.75015133, 0.1047422 , 0.50404393,
       0.88859911, 0.45765575, 0.86455265, 0.60759529, 0.57714447,
       0.99245   , 0.03605465, 0.24410114, 0.32328553, 0.25722123,
       0.18727493, 0.18139005, 0.94100615, 0.82799455, 0.61695013,
       0.82705155, 0.22017298, 0.64696034, 0.0548706 , 0.50184007,
       0.97231793, 0.51327057, 0.06973444, 0.67840464, 0.00348309,
       0.11073477, 0.44336656, 0.57032457, 0.09260419, 0.70727527,
       0.93214017, 0.62784678, 0.7140524 , 0.0608119 , 0.99242956,
       0.32691443, 0.35244788])
    
    def ansatz(Parameters):
        qc = QuantumCircuit(N)
        for d in range(depth):
            param1=Parameters[d*9*N:(d+1)*(9*N)]
            for q in range(N):
                qc.ry(param1[q],qubits[q])
                qc.ry(param1[q+N],qubits[q])
                qc.ry(param1[q+2*N],qubits[q])
            qc.barrier()
            for q in range(N):
                qc.cx(qubits[q], qubits[(q+1)% N])
                qc.ry(param1[q+3*N],qubits[q])
                qc.ry(param1[q+4*N],qubits[(q+1)% N])
                qc.cx(qubits[(q+1)% N], qubits[q])
                qc.ry(param1[q+5*N],qubits[(q+1)% N])
                qc.cx(qubits[q], qubits[(q+1)% N])
            qc.barrier()
            if d==depth-1:
                for q in range(N):
                    qc.ry(param1[q+6*N],qubits[q])
                    qc.ry(param1[q+7*N],qubits[q])
                    qc.ry(param1[q+8*N],qubits[q])
            qc.barrier()
        
        return qc

    def circ(parameters):
        x=QuantumRegister(12)
        c=ClassicalRegister(8)
        circuit = QuantumCircuit(x,c)
    
        circuit=circuit.compose(ansatz(parameters),x[8:12])
        circuit.h(x[0:8]) 
        circuit.append(C_u1gate, [x[0],x[8],x[9],x[10],x[11]])    
        circuit.append(C_u2gate, [x[1],x[8],x[9],x[10],x[11]])    
        circuit.append(C_u3gate, [x[2],x[8],x[9],x[10],x[11]])    
        circuit.append(C_u4gate, [x[3],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u5gate, [x[4],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u6gate, [x[5],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u7gate, [x[6],x[8],x[9],x[10],x[11]]) 
        circuit.append(C_u8gate, [x[7],x[8],x[9],x[10],x[11]]) 
        circuit &= QFT(num_qubits=8, approximation_degree=0, do_swaps=True, 
                       inverse=True, insert_barriers=False, name='qft')
        circuit.measure(x[0:8],c)       
        return circuit

    def cost(Parameters):
        # qct = transpile(qc,basis_gates = ['cx','id', 'x', 'rx', 'rz'])
        # Noise_sim = AerSimulator.from_backend(backend)
        job = backend.run(pm.run(circ(Parameters)),shots = shots).result()
        result = job.get_counts(0)
        if '00000000'not in result: 
            res = 1
        else:
            res = 1 - result['00000000']/shots
        return res
    # cost(parameters)

    def Optimizer(fun, x0, args=(), maxfev=None, 
                  reset_interval=None, eps=None, callback=None, **_):
        
        x0 = np.asarray(x0)
        recycle_z0 = None
        niter = 0
        funcalls = 0
    
        while True:
    
            idx = niter % x0.size
    
            if reset_interval > 0:
                if niter % reset_interval == 0:
                    recycle_z0 = None
    
            if recycle_z0 is None:
                z0 = fun(np.copy(x0), *args)
                funcalls += 1
            else:
                z0 = recycle_z0
    
            p = np.copy(x0)
            p[idx] = x0[idx] + np.pi / 2
            z1 = fun(p, *args)
            funcalls += 1
    
            p = np.copy(x0)
            p[idx] = x0[idx] - np.pi / 2
            z3 = fun(p, *args)
            funcalls += 1
    
            z2 = z1 + z3 - z0
            c = (z1 + z3) / 2
            a = np.sqrt((z0 - z2) ** 2 + (z1 - z3) ** 2) / 2
            b = np.arctan((z1 - z3) / ((z0 - z2) + 1e-32 * (z0 == z2))) + x0[idx]
            b += 0.5 * np.pi + 0.5 * np.pi * np.sign((z0 - z2) + eps * (z0 == z2))
            x0[idx] = b
            recycle_z0 = c - a
            if callback is not None:
                callback(np.copy(x0))
            if funcalls >= maxfev:
                break
            niter += 1
        # return OptimizeResult(fun=problabel0(np.copy(x0)), x=x0, nit=niter, 
        #                       nfev=funcalls, success=(niter > 1))
    
    def save(Parameters):
        global Cost,Params
        Cost.append(cost(Parameters))
        Params.append(Parameters)
        # print(cost(Parameters))
    Cost = []
    Params = []
    Optimizer(cost, Parameters, args=(), maxfev = 4000, 
              reset_interval = 32, eps=1e-32, callback=save)

    e = []
    F = []
    norm_e = []
    for k in range(len(Params)):
        state = np.array(Statevector(ansatz(Params[k])))
        norm = np.dot(state,x_exact)
        e.append(x_exact - state/norm)
        f = abs(np.dot(state,x_exact))**2
        F.append(f)

    for v in e:
        norm_e.append(float(np.linalg.norm(v)))
    Res = norm_e[np.argmax(F)]
    print(Res)
    RMSE.append(Res)

/tmp/ipykernel_870679/900020372.py:61: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit &= QFT(num_qubits=8, approximation_degree=0, do_swaps=True,


0.09967562051294877
0.02775565170847619
0.025778189742145685
0.019660281072207023


In [10]:
print(RMSE)

[0.09967562051294877, 0.02775565170847619, 0.025778189742145685, 0.019660281072207023]


In [ ]:
[0.09967562051294877, 0.02775565170847619, 0.025778189742145685, 0.019660281072207023]
